# Pulser-Gym: Interactive Demo

This demonstration illustrates the integration of classical Deep Reinforcement Learning (DRL) with neutral atom quantum processing using Pasqal's `Pulser` library and OpenAI `Gymnasium`.

## 1. Setup & Module Initialization
Run the following cell to initialize the environment. If you are on **Google Colab**, this will automatically clone the repository and install all required libraries.

In [ ]:
import os
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt

# Suppress internal system and deprecation warnings for a cleaner output
warnings.filterwarnings("ignore")

# --- Environment Setup (Colab / Local) ---
if 'google.colab' in str(get_ipython()):
    # Clone repository if needed
    if not os.path.exists('/content/Pulser-gym'):
        print("Cloning repository...")
        !git clone -q https://github.com/charleshusson75-cell/Pulser-gym.git /content/Pulser-gym
    
    # Ensure we are in the repo directory
    os.chdir('/content/Pulser-gym')
    
    # Install core dependencies silently (including tqdm/rich for progress bars)
    print("Installing dependencies (Pulser, Gymnasium, Stable-Baselines3)...")
    !pip install -q qutip pulser gymnasium stable-baselines3 shimmy tqdm rich

# Register the local environment package path
env_package_path = os.path.abspath('pulser-gym-env')
if env_package_path not in sys.path:
    sys.path.append(env_package_path)

print(f"Active Directory: {os.getcwd()}")

# --- Core Imports ---
from pulser import Register
from stable_baselines3 import PPO
from pulser_gym.env_01_core import PulserEnv
from pulser_gym.sequence_02_translation import build_pulse_sequence

print("✅ Framework and Physical Bridge initialized successfully.")

## 2. Register Layout and Visualization
The 2D square lattice consists of 9 atoms arranged in a 3x3 grid with 5 $\mu$m spacing. The agent optimizes the state selection subject to the Rydberg blockade radius constraints.

In [ ]:
n_atoms = 9
side_length = int(np.sqrt(n_atoms))
graph_register = Register.square(side_length, spacing=5.0, prefix='q')

fig, ax = plt.subplots(figsize=(6, 4))
graph_register.draw(with_labels=True, custom_ax=ax)
ax.set_title(f"2D Lattice Topology ({side_length}x{side_length})")
plt.show()

## 3. Gymnasium Environment Configuration
The environment defines a continuous action space for pulse amplitude scaling and a discrete observation space corresponding to the marginal probability vector of the qubit states.

In [ ]:
env = PulserEnv(n_qubits=n_atoms)

print(f"Action Space: {env.action_space}")
print(f"Observation Space: {env.observation_space}")

## 4. Training the PPO Agent
The agent is trained using Proximal Policy Optimization (PPO) for 1,024 timesteps to maximize the Maximum Independent Set (MIS) objective.

In [ ]:
print("Commencing training loop...")
model = PPO("MlpPolicy", env, verbose=0)
model.learn(total_timesteps=1024, progress_bar=True)

print("\nTraining complete.")

## 5. Evaluation and Waveform Generation
Deterministic inference is performed to extract the optimal laser amplitude and verify the produced quantum state against the graph constraints.

In [ ]:
obs, _ = env.reset()
action, _ = model.predict(obs, deterministic=True)

generated_sequence = build_pulse_sequence(n_atoms, action)
amplitude = action[0] * 10.0
final_obs, final_reward, _, _, _ = env.step(action)

print(f"Selected Amplitude: {amplitude:.3f} rad/us")
print(f"Output State Bitstring: {final_obs}")
print(f"Final MIS Reward: {final_reward:.1f}")

# Note: draw() automatically renders in notebooks in Pulser 1.7+
generated_sequence.draw()